In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from xgboost import XGBRegressor

pd.set_option('display.max_columns', None)

# ============================================================
# LOAD & CLEAN
# ============================================================
df = pd.read_csv('insurance_claims.csv')
if '_c39' in df.columns:
    df = df.drop(columns=['_c39'])
df = df.replace('?', np.nan)
df['fraud_label'] = df['fraud_reported'].map({'Y': 1, 'N': 0})

# ============================================================
# FEATURE ENGINEERING
# ============================================================
drop_cols = ['policy_number', 'policy_bind_date', 'insured_zip', 'incident_date',
             'incident_location', 'auto_model']
df_fe = df.drop(columns=[c for c in drop_cols if c in df.columns])

for col in ['collision_type', 'authorities_contacted', 'property_damage', 'police_report_available']:
    df_fe[col] = df_fe[col].fillna('Unknown')

df_fe['multi_vehicle_incident'] = (df_fe['number_of_vehicles_involved'] >= 2).astype(int)
df_fe['severity_vehicle_interaction'] = (
    df_fe['incident_severity'].astype(str) + "_" + df_fe['multi_vehicle_incident'].astype(str)
)

# --- Severity feature set (leakage-safe) ---
severity_leakage_cols = ['injury_claim', 'property_claim', 'vehicle_claim',
                          'fraud_reported', 'fraud_label', 'total_claim_amount']
X_severity = df_fe.drop(columns=[c for c in severity_leakage_cols if c in df_fe.columns])
y_severity = df_fe['total_claim_amount']

# --- Fraud feature set (hobbies removed — confirmed artifact) ---
fraud_leakage_cols = ['fraud_reported', 'fraud_label', 'insured_hobbies']
X_fraud = df_fe.drop(columns=[c for c in fraud_leakage_cols if c in df_fe.columns])
y_fraud = df_fe['fraud_label']

# ============================================================
# TRAIN/TEST SPLITS
# ============================================================
X_sev_train, X_sev_test, y_sev_train, y_sev_test = train_test_split(
    X_severity, y_severity, test_size=0.2, random_state=42)

X_frd_train, X_frd_test, y_frd_train, y_frd_test = train_test_split(
    X_fraud, y_fraud, test_size=0.2, random_state=42, stratify=y_fraud)

# ============================================================
# ENCODING
# ============================================================
def build_preprocessor(X, scale_numeric=False):
    cat_cols = X.select_dtypes(include='object').columns.tolist()
    num_cols = X.select_dtypes(include=[np.number]).columns.tolist()
    num_transform = StandardScaler() if scale_numeric else 'passthrough'
    preprocessor = ColumnTransformer(
        transformers=[
            ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_cols),
            ('num', num_transform, num_cols)
        ]
    )
    return preprocessor

sev_preprocessor = build_preprocessor(X_sev_train, scale_numeric=False)
X_sev_train_enc = sev_preprocessor.fit_transform(X_sev_train)
X_sev_test_enc = sev_preprocessor.transform(X_sev_test)

frd_preprocessor = build_preprocessor(X_frd_train, scale_numeric=True)
X_frd_train_enc = frd_preprocessor.fit_transform(X_frd_train)
X_frd_test_enc = frd_preprocessor.transform(X_frd_test)

# ============================================================
# TRAIN FINAL MODELS
# ============================================================
xgb_model = XGBRegressor(n_estimators=200, max_depth=4, learning_rate=0.05, random_state=42)
xgb_model.fit(X_sev_train_enc, y_sev_train)
xgb_preds = xgb_model.predict(X_sev_test_enc)

logreg_v2 = LogisticRegression(class_weight='balanced', max_iter=2000, solver='liblinear', random_state=42)
logreg_v2.fit(X_frd_train_enc, y_frd_train)
logreg_v2_probs = logreg_v2.predict_proba(X_frd_test_enc)[:, 1]

print("Models retrained. Severity test shape:", X_sev_test_enc.shape,
      "| Fraud test shape:", X_frd_test_enc.shape)

# ============================================================
# ASSUMPTIONS
# ============================================================
MANUAL_REVIEW_COST = 500  # flat cost per claim flagged for fraud review (investigator time)

# ============================================================
# PART 1: FRAUD MODEL — DOLLAR IMPACT AT TWO THRESHOLDS
# ============================================================
test_claim_amounts = df_fe.loc[X_frd_test.index, 'total_claim_amount']

def calculate_fraud_impact(probs, y_true, claim_amounts, threshold):
    preds = (probs >= threshold).astype(int)

    tp_mask = (preds == 1) & (y_true == 1)
    fn_mask = (preds == 0) & (y_true == 1)
    fp_mask = (preds == 1) & (y_true == 0)

    fraud_caught_amount = claim_amounts[tp_mask].sum()
    fraud_missed_amount = claim_amounts[fn_mask].sum()
    false_alarm_count = fp_mask.sum()
    review_cost = false_alarm_count * MANUAL_REVIEW_COST

    baseline_fraud_loss = claim_amounts[y_true == 1].sum()
    net_benefit_vs_baseline = fraud_caught_amount - review_cost

    return {
        'threshold': threshold,
        'fraud_caught_count': tp_mask.sum(),
        'fraud_caught_amount': fraud_caught_amount,
        'fraud_missed_count': fn_mask.sum(),
        'fraud_missed_amount': fraud_missed_amount,
        'false_alarms': false_alarm_count,
        'review_cost': review_cost,
        'baseline_fraud_loss_if_no_model': baseline_fraud_loss,
        'net_benefit_vs_no_model': net_benefit_vs_baseline
    }

impact_balanced = calculate_fraud_impact(logreg_v2_probs, y_frd_test.values, test_claim_amounts.values, threshold=0.55)
impact_high_recall = calculate_fraud_impact(logreg_v2_probs, y_frd_test.values, test_claim_amounts.values, threshold=0.20)

impact_df = pd.DataFrame([impact_balanced, impact_high_recall])
print("\n=== Fraud Model: Dollar Impact by Threshold ===")
print(impact_df.T)

# ============================================================
# PART 2: SEVERITY MODEL — RESERVE IMPACT VS NAIVE BASELINE
# ============================================================
naive_reserve_per_claim = y_sev_train.mean()
naive_reserve_preds = np.full_like(y_sev_test, naive_reserve_per_claim, dtype=float)

naive_total_error = np.abs(y_sev_test.values - naive_reserve_preds).sum()
model_total_error = np.abs(y_sev_test.values - xgb_preds).sum()

naive_rmse = np.sqrt(np.mean((y_sev_test.values - naive_reserve_preds) ** 2))
model_rmse = np.sqrt(np.mean((y_sev_test.values - xgb_preds) ** 2))

print("\n=== Severity Model: Reserve Accuracy vs Naive Baseline ===")
print(f"Naive baseline (flat average reserve): ${naive_reserve_per_claim:,.0f} per claim")
print(f"Naive baseline RMSE: ${naive_rmse:,.0f}")
print(f"Model RMSE: ${model_rmse:,.0f}")
print(f"RMSE improvement: {(1 - model_rmse/naive_rmse)*100:.1f}%")
print(f"\nTotal absolute reserving error - naive: ${naive_total_error:,.0f}")
print(f"Total absolute reserving error - model: ${model_total_error:,.0f}")
print(f"Reduction in total reserving error: ${naive_total_error - model_total_error:,.0f} "
      f"({(1 - model_total_error/naive_total_error)*100:.1f}% reduction)")

n_test = len(y_sev_test)
avg_error_reduction_per_claim = (naive_total_error - model_total_error) / n_test
print(f"\nAverage reserving error reduction per claim: ${avg_error_reduction_per_claim:,.0f}")
print(f"Projected impact on a 10,000-claim annual book: ${avg_error_reduction_per_claim * 10000:,.0f}")

C:\Users\Anusha\AppData\Local\Temp\ipykernel_5860\1159654495.py:59: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = X.select_dtypes(include='object').columns.tolist()
C:\Users\Anusha\AppData\Local\Temp\ipykernel_5860\1159654495.py:59: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/us

Models retrained. Severity test shape: (200, 126) | Fraud test shape: (200, 110)

=== Fraud Model: Dollar Impact by Threshold ===
                                          0          1
threshold                              0.55        0.2
fraud_caught_count                    33.00       42.0
fraud_caught_amount              2074560.00  2625060.0
fraud_missed_count                    16.00        7.0
fraud_missed_amount               923910.00   373410.0
false_alarms                          21.00       94.0
review_cost                        10500.00    47000.0
baseline_fraud_loss_if_no_model  2998470.00  2998470.0
net_benefit_vs_no_model          2064060.00  2578060.0

=== Severity Model: Reserve Accuracy vs Naive Baseline ===
Naive baseline (flat average reserve): $53,283 per claim
Naive baseline RMSE: $25,928
Model RMSE: $14,753
RMSE improvement: 43.1%

Total absolute reserving error - naive: $4,013,666
Total absolute reserving error - model: $2,187,140
Reduction in total reservin